# NumPy Deep Dive: From Basics to Vectorized Operations

NumPy is the foundational library for numerical computing in Python. Every major data science and machine learning framework—pandas, scikit-learn, TensorFlow, PyTorch—builds on NumPy arrays under the hood. This notebook provides a thorough exploration of NumPy, from array creation through advanced vectorized operations, with real-world contexts throughout.


## Why NumPy?

Standard Python lists are flexible but slow for numerical work. NumPy arrays are:
- **Homogeneous**: all elements share the same dtype, enabling contiguous memory layout
- **Vectorized**: operations apply to entire arrays without explicit loops
- **Memory-efficient**: compact C-level storage with no Python object overhead per element

Mastering NumPy is the single highest-leverage skill for writing fast, readable data science code.


---
## 1. Creating Arrays

NumPy provides many factory functions to create arrays from scratch or convert existing data.


In [ ]:
import numpy as np

# From a list
arr_from_list = np.array([1, 2, 3, 4, 5])
print("From list:", arr_from_list)

# Zeros (useful for initializing weights in ML)
zeros = np.zeros((3, 4))
print("Zeros (3x4):\n", zeros)

# Ones
ones = np.ones((2, 3))
print("Ones (2x3):\n", ones)

# Identity matrix
eye = np.eye(4)
print("Identity (4x4):\n", eye)

# arange — like range() but returns array
arange_arr = np.arange(0, 10, 2)
print("arange(0, 10, 2):", arange_arr)

# linspace — evenly spaced numbers over interval
linspace_arr = np.linspace(0, 1, 5)
print("linspace(0, 1, 5):", linspace_arr)

# Random arrays
rand_uniform = np.random.rand(2, 3)
print("Uniform random (2x3):\n", rand_uniform)

# Full — fill with a constant
full_arr = np.full((2, 3), 7)
print("Full of 7s (2x3):\n", full_arr)


### Real-world context: image as a 3D array

A color image is a 3D array with shape (height, width, channels). We simulate a small 4x4 RGB image.


In [ ]:
# Simulate an RGB image: 4x4 pixels, 3 color channels
image = np.random.randint(0, 256, size=(4, 4, 3), dtype=np.uint8)
print("Simulated image shape:", image.shape)
print("Red channel (top-left 4x4):\n", image[:, :, 0])
print("Pixel at row 1, col 2 (R,G,B):", image[1, 2, :])


## 2. Array Attributes

Every ndarray has a set of attributes that describe its structure and memory layout.


In [ ]:
arr = np.array([[1.5, 2.3, 3.9],
                [4.1, 5.7, 6.2],
                [7.8, 8.4, 9.0]])

print("Data:\n", arr)
print("shape:", arr.shape)
print("dtype:", arr.dtype)
print("size (total elements):", arr.size)
print("ndim (number of axes):", arr.ndim)
print("itemsize (bytes per element):", arr.itemsize)
print("nbytes (total bytes):", arr.nbytes)
print("strides (bytes to step per axis):", arr.strides)


**Understanding strides**: Strides tell NumPy how many bytes to skip to move one element along each axis. Row-major (C-order) means the last axis changes fastest.


In [ ]:
# Changing dtype changes array interpretation
int_arr = np.array([1, 2, 3], dtype=np.int32)
float_arr = int_arr.astype(np.float64)
print("int32 itemsize:", int_arr.itemsize)
print("float64 itemsize:", float_arr.itemsize)

# Note: astype creates a copy — be mindful with large arrays


## 3. Indexing and Slicing

NumPy offers powerful indexing that goes well beyond Python lists.


### Basic slicing (views, not copies)


In [ ]:
arr_2d = np.arange(20).reshape(4, 5)
print("Original:\n", arr_2d)

# Row and column slicing
print("First 2 rows, columns 1:3:\n", arr_2d[:2, 1:3])

# Step slicing
print("Every other row, every other col:\n", arr_2d[::2, ::2])

# Slices are VIEWS — modifying them affects the original
view = arr_2d[0:2, 0:2]
view[:] = 99
print("After modifying view:\n", arr_2d)


### Fancy indexing (integer arrays)


In [ ]:
arr = np.arange(10)
indices = np.array([3, 7, 1, 9])
print("Elements at indices [3,7,1,9]:", arr[indices])

# 2D fancy indexing — select specific rows
mat = np.arange(25).reshape(5, 5)
rows = np.array([0, 2, 4])
print("Rows 0, 2, 4:\n", mat[rows])

# Combined: specific (row, col) pairs
print("Elements at (0,1), (2,3), (4,0):",
      mat[[0, 2, 4], [1, 3, 0]])


### Boolean masking


In [ ]:
data = np.random.randn(10)
print("Data:", data)

# Mask for positive values
mask = data > 0
print("Mask:", mask)
print("Positive values:", data[mask])

# Replace all negative values with 0
data_clipped = data.copy()
data_clipped[data_clipped < 0] = 0
print("After clipping negatives:\n", data_clipped)

# Boolean masking is widely used in ML for outlier removal
outlier_mask = np.abs(data) > 2 * data.std()
print("Outlier count:", outlier_mask.sum(), "out of", len(data))


## 4. Reshaping and Broadcasting

Reshaping changes an array's dimensions without altering its data. Broadcasting is NumPy's system for performing operations between arrays of different shapes.


### Reshaping


In [ ]:
arr = np.arange(12)
print("1D:", arr)

# Reshape to 3x4
reshaped = arr.reshape(3, 4)
print("3x4:\n", reshaped)

# Flatten back (ravel returns view when possible)
flat = reshaped.ravel()
print("Raveled:", flat)

# Reshape with -1 (infer dimension)
auto = arr.reshape(2, -1)
print("Auto-inferred (2, 6):\n", auto)

# Adding / removing singleton dimensions
col_vec = np.arange(5).reshape(-1, 1)
print("Column vector shape:", col_vec.shape)
print("Column vector:\n", col_vec)

row_vec = np.arange(5).reshape(1, -1)
print("Row vector shape:", row_vec.shape)


### Broadcasting rules

When operating on two arrays, NumPy compares their shapes **element-wise from right to left**. Two dimensions are compatible when:
1. They are equal, or
2. One of them is 1

If neither condition holds, broadcasting fails.

**Visual diagram:**
```
Array A:  (3, 1, 4)
Array B:  (   2, 4)   ->  broadcast to (1, 2, 4)
Result:   (3, 2, 4)
```


In [ ]:
# Scalars broadcast to whole array
arr = np.array([1, 2, 3])
print("arr + 10:", arr + 10)

# Row + column vector broadcasting
row = np.array([10, 20, 30])
col = np.array([[1], [2], [3]])
print("Row shape:", row.shape)
print("Col shape:", col.shape)
print("Row + Col:\n", row + col)

# Example: centering a dataset (subtract column means)
data_2d = np.random.randn(5, 3)
col_means = data_2d.mean(axis=0)
centered = data_2d - col_means
print("After centering, column means are near zero:",
      centered.mean(axis=0).round(10))


### Broadcasting error example


In [ ]:
# Uncommenting this would raise ValueError:
a = np.ones((3, 4))
b = np.ones((2, 5))
# c = a + b  # shapes (3,4) and (2,5) not compatible
print("Expected: Broadcasting error — dimensions 4 vs 5 and 3 vs 2 both mismatch")


## 5. Universal Functions and Vectorization

Universal functions (ufuncs) operate element-wise on arrays. They are implemented in compiled C and are far faster than Python loops.


### Common ufuncs


In [ ]:
arr = np.array([0, np.pi/4, np.pi/2, np.pi])

print("Input:", arr)
print("sin:", np.sin(arr))
print("cos:", np.cos(arr))
print("exp:", np.exp(arr[:3]))
print("log:", np.log(np.array([1, 2, 3, 4, 5])))
print("sqrt:", np.sqrt(np.array([1, 4, 9, 16])))
print("abs:", np.abs(np.array([-3, -1, 0, 2, 5])))


### Performance comparison: NumPy vectorization vs Python loop

We'll use `%%timeit` to compare element-wise operations on a large array.


In [ ]:
%%timeit
size = 10_000_000
arr = np.random.randn(size)
# Vectorized: compute element-wise sin, square, add constant
result = np.sin(arr) ** 2 + np.cos(arr) ** 2
# This should equal 1 for all elements (trig identity)


In [ ]:
%%timeit
size = 10_000_000
# Python list version
py_list = list(np.random.randn(size).tolist())
result_py = [(np.sin(x) ** 2 + np.cos(x) ** 2) for x in py_list]
# The NumPy version is typically 50-100x faster


### Reducing functions (aggregation ufuncs)


In [ ]:
arr = np.arange(1, 13).reshape(3, 4)
print("Array:\n", arr)

print("Sum all:", arr.sum())
print("Sum axis=0 (down rows):", arr.sum(axis=0))
print("Sum axis=1 (across cols):", arr.sum(axis=1))

print("Cumulative sum:", np.cumsum(np.arange(1, 6)))
print("Cumulative product:", np.cumprod(np.arange(1, 6)))


## 6. Linear Algebra Operations

NumPy's `linalg` module provides BLAS/LAPACK-backed linear algebra routines essential for ML algorithms.


In [ ]:
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])

# Dot product (matrix multiplication)
print("A @ B:\n", A @ B)
print("np.dot(A, B):\n", np.dot(A, B))
print("np.matmul(A, B):\n", np.matmul(A, B))

# Transpose
print("A transpose:\n", A.T)

# Matrix inverse
A_inv = np.linalg.inv(A)
print("A inverse:\n", A_inv)

# Verify: A @ A_inv ≈ identity
print("A @ A_inv:\n", (A @ A_inv).round(2))


### Eigenvalues and eigenvectors


In [ ]:
# Eigen decomposition (useful for PCA)
eigvals, eigvecs = np.linalg.eig(A)
print("Eigenvalues:", eigvals)
print("Eigenvectors (columns):\n", eigvecs)

# Verify: A @ v = lambda * v
v = eigvecs[:, 0]
lam = eigvals[0]
print("A @ v:", A @ v)
print("lambda * v:", lam * v)


### Singular Value Decomposition (SVD)


In [ ]:
# SVD: A = U @ S @ Vh  (core of PCA, matrix factorization, recommender systems)
U, S, Vh = np.linalg.svd(A)
print("U shape:", U.shape)
print("Singular values:", S)
print("Vh shape:", Vh.shape)

# Reconstruct (S must be diagonalized)
S_mat = np.zeros((2, 2))
np.fill_diagonal(S_mat, S)
A_reconstructed = U @ S_mat @ Vh
print("Reconstructed A:\n", A_reconstructed.round(2))


### Solving linear systems


In [ ]:
# Solve Ax = b
A = np.array([[3, 1],
              [1, 2]])
b = np.array([9, 8])

x = np.linalg.solve(A, b)
print("Solution x:", x)

# Verify
print("A @ x =", A @ x)
print("b     =", b)


## 7. Statistical Operations

NumPy provides efficient statistical aggregations commonly used in data exploration and ML preprocessing.


In [ ]:
# Simulate a tabular dataset: 1000 samples, 5 features
np.random.seed(42)
data = np.random.randn(1000, 5)

print("Shape:", data.shape)
print("Mean per feature:", data.mean(axis=0).round(4))
print("Std per feature:", data.std(axis=0).round(4))
print("Variance per feature:", data.var(axis=0).round(4))

print("Min per feature:", data.min(axis=0).round(4))
print("Max per feature:", data.max(axis=0).round(4))
print("Argmin (row index of min) per feature:", data.argmin(axis=0))
print("Argmax (row index of max) per feature:", data.argmax(axis=0))


In [ ]:
# Percentiles and quartiles
flat_data = np.random.randn(10000)
percentiles = np.percentile(flat_data, [25, 50, 75, 95, 99])
print("25th, 50th, 75th, 95th, 99th percentiles:", percentiles.round(4))

# Median is the 50th percentile
print("Median:", np.median(flat_data).round(4))

# Standardize features (z-score normalization for ML)
mean = data.mean(axis=0)
std = data.std(axis=0)
data_standardized = (data - mean) / std
print("After standardization — mean near 0:", data_standardized.mean(axis=0).round(10))
print("After standardization — std near 1:", data_standardized.std(axis=0).round(4))


## 8. Random Module

NumPy's `random` module is the workhorse for generating synthetic data, shuffling, and sampling in ML pipelines.


In [ ]:
# Set seed for reproducibility
np.random.seed(42)

# Uniform distribution [0, 1)
uniform = np.random.rand(3, 3)
print("Uniform (3x3):\n", uniform)

# Standard normal distribution
normal = np.random.randn(3, 3)
print("Standard normal (3x3):\n", normal)

# Random integers
ints = np.random.randint(0, 10, size=(2, 4))
print("Random integers [0, 10):\n", ints)

# Choice with and without replacement
population = np.arange(10)
sample = np.random.choice(population, size=5, replace=False)
print("Sample without replacement:", sample)

# Shuffle in-place
deck = np.arange(1, 11)
np.random.shuffle(deck)
print("Shuffled deck:", deck)

# Different distributions
binomial = np.random.binomial(n=10, p=0.5, size=5)
print("Binomial (n=10, p=0.5):", binomial)

exponential = np.random.exponential(scale=1.0, size=5)
print("Exponential (scale=1.0):", exponential.round(4))


## 9. Advanced Broadcasting and Einsum

### Broadcasting rules — detailed shape diagrams

Rule: NumPy compares shapes **right to left**. At each dimension:
- Same size → compatible
- One is 1 → broadcast (stretch)
- Otherwise → error

**Examples:**
```
A shape:  (5, 1, 3)
B shape:     (4, 3)
                ^  ^
                3==3 ok, 1 vs 4 → broadcast to 4
Result:   (5, 4, 3)

A shape:  (3, 4, 5)
B shape:     (4, 1)
Result:   (3, 4, 5)   — 5 vs 1 ok, 4==4 ok, 3 missing → broadcast
```


In [ ]:
# Practical ML example: adding bias terms
# Batch of 8 samples, each with 3 features
batch = np.random.randn(8, 3)
# Bias vector (1D) broadcasts across rows
bias = np.array([0.1, -0.2, 0.05])
biased = batch + bias
print("Batch shape:", batch.shape, "| Bias shape:", bias.shape)
print("Result shape:", biased.shape)
print("First sample original:", batch[0].round(2))
print("First sample biased:", biased[0].round(2))


### Einsum — Einstein summation

Einsum provides a compact notation for complex tensor operations: matrix multiplication, transpose, dot products, batch operations, and more — all in one function.


In [ ]:
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])

# Matrix multiplication via einsum
C = np.einsum('ij,jk->ik', A, B)
print("einsum matmul:\n", C)
print("Same as A @ B:\n", A @ B)

# Dot product of two vectors
v = np.array([1, 2, 3])
w = np.array([4, 5, 6])
print("einsum dot:", np.einsum('i,i->', v, w))
print("np.dot:", np.dot(v, w))

# Transpose
print("einsum transpose:\n", np.einsum('ij->ji', A))

# Batch matrix multiplication (useful for neural nets)
batched_A = np.random.randn(4, 3, 2)
batched_B = np.random.randn(4, 2, 5)
result = np.einsum('bij,bjk->bik', batched_A, batched_B)
print("Batch matmul shape:", result.shape)


## 10. Performance Tips

Writing efficient NumPy code requires understanding memory layout and avoiding unnecessary copies.


### Memory layout

- **C-order (row-major)**: default. Last axis changes fastest. Contiguous in memory.
- **F-order (Fortran/column-major)**: first axis changes fastest.
- Operating on contiguous memory is faster because of CPU cache locality.


In [ ]:
arr = np.random.randn(5000, 5000)
print("Default order (C-contiguous):", arr.flags['C_CONTIGUOUS'])

# Summing along contiguous axis is faster
get_ipython().run_line_magic('timeit', 'arr.sum(axis=1)')

# Transpose makes non-contiguous copy (or view)
arr_t = arr.T
print("Transpose is not C-contiguous:", arr_t.flags['C_CONTIGUOUS'])
get_ipython().run_line_magic('timeit', 'arr_t.sum(axis=1)')


### In-place operations


In [ ]:
# In-place operations avoid allocating new arrays
arr = np.random.randn(1000, 1000)
arr_copy = arr.copy()

# Non-in-place: creates a new array
print("Non-in-place (arr + 5):", end=" ")
get_ipython().run_line_magic('timeit', 'arr + 5')

# In-place: modifies and returns None
print("In-place (arr_copy += 5):", end=" ")
get_ipython().run_line_magic('timeit', 'arr_copy += 5')


### Avoiding copies

- `.copy()` creates a new array — use it only when you need independent data
- `.view()` creates a new array object sharing the same data
- Slicing returns views (not copies)
- Fancy indexing and boolean masking return copies


In [ ]:
arr = np.arange(10)

# Slice → view (no copy)
view = arr[2:5]
view[0] = 999
print("After modifying view, orig:", arr)

# Fancy index → copy
copy = arr[[1, 3, 5]]
copy[0] = -100
print("After modifying fancy index copy, orig:", arr)

# Boolean mask → copy
masked = arr[arr > 5]
masked[0] = -1
print("After modifying masked copy, orig:", arr)

# Use np.shares_memory to check
print("Slice shares memory?",
      np.shares_memory(arr, arr[2:5]))
print("Fancy index shares memory?",
      np.shares_memory(arr, arr[[1, 3, 5]]))


### Pre-allocating vs appending


In [ ]:
# BAD: appending in a loop (each append creates a new array)
def grow_badly(n):
    result = np.array([])
    for i in range(n):
        result = np.append(result, i)
    return result

# GOOD: pre-allocate and assign
def grow_well(n):
    result = np.empty(n)
    for i in range(n):
        result[i] = i
    return result

print("Bad approach (appending):", end=" ")
get_ipython().run_line_magic('timeit', 'grow_badly(10000)')
print("Good approach (pre-allocate):", end=" ")
get_ipython().run_line_magic('timeit', 'grow_well(10000)')


## 11. Try It Yourself — Exercises

Apply what you've learned to these practical problems.


### Exercise 1: Feature scaling

Create a 100x4 array of random integers between 0 and 1000. Min-max normalize it so each column ranges from 0 to 1.


In [ ]:
# Your code here
import numpy as np

# Step 1: create random data (100 samples, 4 features)
# Step 2: compute column-wise min and max
# Step 3: apply (X - min) / (max - min) using broadcasting


### Exercise 2: Image grayscale conversion

Given a simulated 10x10 RGB image (3D array with shape 10x10x3), convert it to grayscale using the luminance weights: `gray = 0.299*R + 0.587*G + 0.114*B`.


In [ ]:
# Your code here
image = np.random.randint(0, 256, size=(10, 10, 3), dtype=np.uint8)

# Step 1: apply weights using broadcasting or einsum
# Step 2: verify output shape is (10, 10)


### Exercise 3: Time series moving average

Create a 1D array representing 1000 time steps of a noisy signal. Compute a 5-point moving average (centered) using array slicing and mean. Avoid loops.


In [ ]:
# Your code here
t = np.linspace(0, 10 * np.pi, 1000)
signal = np.sin(t) + 0.5 * np.random.randn(1000)

# Step 1: use slicing to create a 2D array of rolling windows
# Step 2: average across the window axis
# Step 3: compare smoothed vs original


### Exercise 4: Matrix factorization with SVD

Create a 5x4 matrix of rank 3. Compute its SVD and reconstruct a rank-2 approximation.


In [ ]:
# Your code here
# Step 1: create a rank-3 matrix by multiplying 5x3 @ 3x4
# Step 2: compute SVD
# Step 3: reconstruct using only the 2 largest singular values
# Step 4: compare approximation error


### Exercise 5: Batch gradient computation

Given a batch of 32 samples (each with 64 features) and a weight matrix of shape (64, 1), compute the batch prediction `X @ w` using einsum. Then compute the mean squared error against a random target vector of shape (32, 1).


In [ ]:
# Your code here
X = np.random.randn(32, 64)
w = np.random.randn(64, 1)
y = np.random.randn(32, 1)

# Step 1: compute predictions using einsum
# Step 2: compute residuals
# Step 3: compute mean squared error
# Step 4: bonus — compute gradient: 2 * X.T @ (pred - y) / 32


## Summary

In this notebook you learned:
- Creating arrays of various shapes and dtypes
- Indexing: basic slicing, fancy indexing, boolean masking
- Reshaping and the broadcasting rules
- Vectorization with universal functions — orders of magnitude faster than loops
- Linear algebra: matrix operations, eigendecomposition, SVD, solving systems
- Statistical operations for data exploration and preprocessing
- The random module for simulation and sampling
- Advanced topics: broadcasting with shape diagrams, einsum
- Performance tips: memory layout, in-place ops, avoiding copies

NumPy is the engine that powers Python data science. Every concept here directly transfers to pandas, scikit-learn, and deep learning frameworks.
